In [11]:
### 1. Load and Prepare Image
from pathlib import Path
import numpy as np
import tifffile as tiff

# Set your input path here
input_path = Path(r"C:\Users\adima\Desktop\Final_dataset\train\PN_KC_488_568_Brain5_CalyxL_LSMPlus_16bit.ome.tif")

def load_and_standardize(path):
    """Loads a 4D image and standardizes its dimension order to (Z, C, Y, X)."""
    img = tiff.imread(str(path))
    if img.ndim != 4:
        raise ValueError(f"Expected a 4D image, got shape {img.shape}")

    # Standardize to (Z, C, Y, X)
    if img.shape[0] == 2:          # (C, Z, Y, X) -> (Z, C, Y, X)
        img = np.transpose(img, (1, 0, 2, 3))
    elif img.shape[1] == 2:        # (Z, C, Y, X) - already correct
        pass
    else:
        raise ValueError(f"Cannot infer channel axis from shape {img.shape}")

    return img

# Load the full stack
img_stack = load_and_standardize(input_path)
print("Standardized stack shape:", img_stack.shape)


Standardized stack shape: (38, 2, 1101, 1101)


In [12]:
### 2. Apply Rolling Ball Background Subtraction
from skimage.restoration import rolling_ball
from tqdm.notebook import tqdm

print("Applying rolling ball background subtraction to the full stack...")
img_bg_sub = np.zeros_like(img_stack, dtype=np.float32)
ball_radius = 50 # You can adjust this value

# Process each slice in the Z-stack
for z in tqdm(range(img_stack.shape[0]), desc="Z-slices"):
    # Channel 0: Red, Channel 1: Green
    for c, channel_name in enumerate(['Red', 'Green']):
        background = rolling_ball(img_stack[z, c, :, :], radius=ball_radius)
        img_bg_sub[z, c, :, :] = img_stack[z, c, :, :] - background

print("Background subtraction complete for the entire stack.")


Applying rolling ball background subtraction to the full stack...


Z-slices:   0%|          | 0/38 [00:00<?, ?it/s]

Background subtraction complete for the entire stack.


In [13]:
### 3. Universal Normalization Function
from skimage.exposure import rescale_intensity

def normalize_robust(image, p_low=0.0001, p_high=99.9999, out_range=(0.0, 1.0)):
    """
    Robustly normalizes a 4D image (Z, C, Y, X) to a given range using percentiles.
    Normalization is done channel-wise across the entire Z-stack.
    """
    # Ensure image is float for calculations
    img = image.astype(np.float32)

    if img.ndim == 4: # 4D image (Z, C, Y, X)
        normalized_stack = np.zeros_like(img)
        for c in range(img.shape[1]):
            # Calculate percentiles for the current channel across all Z-slices
            channel_data = img[:, c, :, :]
            p_low_val, p_high_val = np.percentile(channel_data, (p_low, p_high))
            
            # Rescale the channel to the output range
            normalized_stack[:, c, :, :] = rescale_intensity(
                channel_data, 
                in_range=(p_low_val, p_high_val), 
                out_range=out_range
            )
        return normalized_stack
    else:
        raise ValueError(f"Unsupported image dimension: {img.ndim}. Expected 4D (Z, C, Y, X).")

print("`normalize_robust` function is now defined and ready to use.")
# Example of use:
normalized_image = normalize_robust(img_bg_sub)


`normalize_robust` function is now defined and ready to use.


In [ ]:
### 4. Deconvolution / Filtering
from skimage.restoration import richardson_lucy
from skimage.filters import gaussian
from tqdm.notebook import tqdm

def clip_background(stack, percentile=2):
    """Clips the lowest percentile of values to 0 for a 4D stack."""
    clipped = stack.copy()
    for c in range(stack.shape[1]): # Iterate over channels
        channel_data = stack[:, c, :, :]
        thresh = np.percentile(channel_data, percentile)
        clipped_channel = np.where(channel_data < thresh, 0, channel_data - thresh)
        clipped[:, c, :, :] = clipped_channel
    return clipped

def richardson_lucy_deconvolution_3d(image_stack):
    """
    Applies Richardson-Lucy deconvolution to each slice of a 4D (Z, C, Y, X) image stack.
    Assumes the input image_stack is already normalized to the [0, 1] range.
    """
    deconv_stack = np.zeros_like(image_stack, dtype=np.float32)
    
    kernel_size = 15
    def make_psf(sigma):
        k = np.zeros((kernel_size, kernel_size))
        k[kernel_size // 2, kernel_size // 2] = 1.0
        psf = gaussian(k, sigma=sigma)
        return psf / psf.sum()
    
    psf_red   = make_psf(sigma=1.75)
    psf_green = make_psf(sigma=1.50)
    
    psfs = [psf_red, psf_green]
    n_iters = [15, 15]

    for z in tqdm(range(image_stack.shape[0]), desc="Deconvolving Z-slices"):
        for c, (name, psf, n_iter) in enumerate(zip(['Red', 'Green'], psfs, n_iters)):
            ch_norm = image_stack[z, c].astype(np.float32)
            result = richardson_lucy(ch_norm, psf, num_iter=n_iter, clip=True)
            deconv_stack[z, c] = result

    print("  - Clipping background artifacts...")
    deconv_stack = clip_background(deconv_stack, percentile=2)

    return deconv_stack

# --- Step 1: Run RL on normalized image stack ---
img_deconvolved = richardson_lucy_deconvolution_3d(normalized_image)

# --- Step 2: Add RL output to normalized original ---
img_combined = normalized_image + img_deconvolved

# --- Step 3: Min-max rescale combined result to [0,1] ---
img_final = np.zeros_like(img_combined)
for c in range(img_combined.shape[1]):
    channel_data = img_combined[:, c, :, :]
    c_min, c_max = channel_data.min(), channel_data.max()
    if c_max > c_min:
        img_final[:, c, :, :] = (channel_data - c_min) / (c_max - c_min)
    else:
        img_final[:, c, :, :] = channel_data

print(f"Final output shape: {img_final.shape}")
print(f"Final output range: {img_final.min():.4f} - {img_final.max():.4f}")


Applying DoG Sharpening:   0%|          | 0/38 [00:00<?, ?it/s]

Final output shape: (38, 2, 1101, 1101)
Final output range: 0.0000 - 1.0000


In [17]:
### 5. Save the Deconvolved Image
import tifffile as tiff

# --- Prepare filename ---
output_filename = input_path.stem + "_dog.tif"
output_path = input_path.parent / output_filename

# --- Save the final 4D stack ---
# Transpose back to C, Z, Y, X for compatibility with image viewers like ImageJ/Fiji
# The expected shape is (channels, depth, height, width)
img_to_save = np.transpose(img_final, (1, 0, 2, 3)) 

# Ensure it's in a format that tifffile can handle well, like float32
img_to_save = img_to_save.astype(np.float32)

print(f"Saving deconvolved stack to: {output_path}")
tiff.imwrite(output_path, img_to_save, imagej=True)
print("Save complete.")

Saving deconvolved stack to: C:\Users\adima\Desktop\Final_dataset\train\PN_KC_488_568_Brain5_CalyxL_LSMPlus_16bit.ome_dog.tif
Save complete.


In [ ]:
# ### 6. Visualization with Napari (optional)
# import napari

# # Select a slice to visualize
# vis_slice_index = 8

# viewer = napari.Viewer()

# # --- Helper for Normalization ---
# def normalize_minmax(data):
#     """Scales data to the [0, 1] range."""
#     min_val = data.min()
#     max_val = data.max()
#     if max_val > min_val:
#         return (data - min_val) / (max_val - min_val)
#     return data # Return as is if flat

# # --- Add Layers for Comparison ---
# # Original Image Slice (Normalized for better viewing)
# original_slice = img_stack[vis_slice_index]
# viewer.add_image(normalize_minmax(original_slice[0]), name=f'Original Slice {vis_slice_index} - Red', colormap='red', blending='additive', visible=False)
# viewer.add_image(normalize_minmax(original_slice[1]), name=f'Original Slice {vis_slice_index} - Green', colormap='green', blending='additive', visible=False)

# # Final Deconvolved Image Slice
# final_slice = img_final[vis_slice_index]
# viewer.add_image(final_slice[0], name=f'Final Slice {vis_slice_index} - Red', colormap='red', blending='additive', visible=True)
# viewer.add_image(final_slice[1], name=f'Final Slice {vis_slice_index} - Green', colormap='green', blending='additive', visible=True)

# # You can also view the full 3D stack
# # viewer.add_image(img_final, channel_axis=1, name=['Final 3D - Red', 'Final 3D - Green'], colormap=['red', 'green'], blending='additive')

# napari.run()
